# Measurement-Mechanistic RCA: A Running Example

Install with `pip install -e ".[causal-discovery,notebooks]"` from the repository root. The core example uses only the base package; the final optional section uses NoGAM.

Graph: `W <- X -> Y -> Z`. Row 0 has a measurement error at Y; row 1 has a mechanism shift at Y. Both violate X -> Y, but only the recording error breaks Y -> Z. The data are illustrative, not the original paper figure data.

## 1. Generate the data

The order matters: change the mechanism before generating descendants; corrupt a measurement after generating descendants. Labels are used only to validate the output, never passed into RCA.

In [1]:
import networkx as nx
import numpy as np

from measurement_mechanistic_rca import MeasurementMechanisticRCA

# Column indices: X=0, Y=1, Z=2, W=3.
names = ["X", "Y", "Z", "W"]
dag = nx.DiGraph([(0, 1), (1, 2), (0, 3)])


def make_data(n, seed, contaminated=False):
    rng = np.random.default_rng(seed)
    x = rng.normal(size=n)
    y = 0.8 * x + rng.normal(scale=0.15, size=n)

    if contaminated:
        y[1] += 4.0  # Mechanism shift at Y, BEFORE generating its child Z.

    z = 1.2 * y + rng.normal(scale=0.15, size=n)
    w = -x + rng.normal(scale=0.15, size=n)

    if contaminated:
        y[0] += 4.0  # Measurement error at Y, AFTER generating Z.

    return np.column_stack([x, y, z, w])


data = make_data(n=500, seed=7, contaminated=True)

![Injected anomalies](assets/running_example.png)

Gray: unmodified rows. Blue: measurement error. Orange: mechanism shift. Colors show the known injected types, not model predictions.

## 2. Fit the contaminated batch

No external priors are supplied. We check that Y ranks first and that its type is correct for both injected anomalies. Scores are not probabilities; these two checks do not imply perfect recovery of all assignments or benchmark-level accuracy.

In [2]:
# The method's internal sampler uses NumPy's global random state.
np.random.seed(7)
result = MeasurementMechanisticRCA().fit_explain(data, dag)


def show_and_check(result):
    for row, expected_kind in [(0, "measurement"), (1, "mechanistic")]:
        cause = result.top_features(sample_idx=row, k=1)[0]
        print(f"row {row}: root cause {names[cause.feature]}, type {cause.anomaly_type}")
        assert cause.feature == 1, f"Wrong root cause for row {row}"
        assert cause.anomaly_type == expected_kind, f"Wrong anomaly type for row {row}"


show_and_check(result)

row 0: root cause Y, type measurement
row 1: root cause Y, type mechanistic


### Inspect a root cause
Each candidate has a feature, anomaly type, root-cause score, and conditional confidence. Confidence compares competing type explanations with the other assignments fixed; it is a cost difference, not a probability. Unresolved origins are returned as `uncertain`; joint measurement and mechanistic assignments as `both`.

In [3]:
cause = result.top_features(sample_idx=0, k=1)[0]
print(names[cause.feature], cause.anomaly_type)
print("Root-cause score:", cause.score)
print("Conditional confidence:", cause.conditional_confidence)
print("Measurement features:", list(result.assignment[0]["measurement"]))
print("Mechanistic features:", list(result.assignment[1]["mechanistic"]))

Y measurement
Root-cause score: 133.04857016919473
Conditional confidence: 38.286001103262514
Measurement features: [np.int64(1)]
Mechanistic features: [np.int64(1)]


## 3. Fit clean data and explain new samples

Clean data identifies normal mechanisms, not the frequency of future interventions. Supply measurement and mechanistic priors per feature. Our illustrative 1% probabilities are equal for every feature, so they do not reveal the injected locations. These are application assumptions, not estimates from the clean data. The default `single_root` prior mode assumes at most one root-cause feature per sample.

In [4]:
# Independent clean training set and a separate future batch.
clean_data = make_data(n=500, seed=8)
new_samples = make_data(n=100, seed=9, contaminated=True)

# Illustrative per-feature probabilities, identical for every feature.
# These express prior beliefs about future anomalies, not fitted frequencies.
measurement_priors = np.full(4, 0.01)
mechanistic_priors = np.full(4, 0.01)

np.random.seed(7)
model = MeasurementMechanisticRCA().fit(clean_data, dag)
future_result = model.explain(
    new_samples,
    measurement_priors=measurement_priors,
    mechanistic_priors=mechanistic_priors,
)
show_and_check(future_result)

row 0: root cause Y, type measurement
row 1: root cause Y, type mechanistic


## 4. Optional: use supplied priors with contaminated data

Priors may also have shape `(n, d)` for sample-specific beliefs. Exhaustive inference is available for small graphs via `prior_mode="exhaustive"` (search size `4^d`).

In [5]:
np.random.seed(7)
result_with_priors = MeasurementMechanisticRCA().fit_explain(
    data,
    dag,
    measurement_priors=measurement_priors,
    mechanistic_priors=mechanistic_priors,
)
show_and_check(result_with_priors)

row 0: root cause Y, type measurement
row 1: root cause Y, type mechanistic


## 5. Optional: causal discovery before RCA

When the graph is unknown, NoGAM can estimate it. A learned graph need not equal the true graph, especially in this linear Gaussian example, whose direction is not identifiable from observational data alone. This section demonstrates the API; the assertions above use the known synthetic graph.

In [6]:
from measurement_mechanistic_rca import discover_dag_nogam

np.random.seed(7)
learned_dag = discover_dag_nogam(clean_data, n_crossval=3)
print("Learned edges:", list(learned_dag.edges()))
discovery_result = MeasurementMechanisticRCA().fit_explain(data, learned_dag)
for row in (0, 1):
    print(row, discovery_result.top_features(sample_idx=row, k=1))

Learned edges: [(np.int64(0), np.int64(3)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(2))]


0 [RootCause(feature=1, anomaly_type='measurement', score=81.08751549520446, conditional_confidence=45.14460120116134)]
1 [RootCause(feature=1, anomaly_type='mechanistic', score=37.59148748278851, conditional_confidence=-3.2582853137098162)]


## Regression backends

The default `expectile` backend uses Python/pyGAM. `quantile` uses scikit-learn spline median regression. `qgam` uses the R backend from the paper experiments and requires the R dependencies. Expectile and median regression are distinct objectives.